# Notebook 12 — Exploratory Full-Backbone Fine-Tuning (Kaggle T4)

**Purpose:** Compare full fine-tuning (v3) against unchanged partial fine-tuning (Notebook 11, v2).
**Priority:** E3-v3 (Stanford/Xception), then E4-v3 (Stanford/ConvNeXt-Tiny).

Both models use ImageNet-pretrained initialization, a new classifier, 5 head-only epochs, then up to 40 full-backbone fine-tuning epochs. BatchNorm running statistics remain in eval mode during training; affine parameters remain trainable in Stage 2. All selection uses validation macro-F1.

**Methodology:** Original test scores and v2 test results have already been inspected. This is an exploratory **post-hoc** experiment. Never describe any v3 test result as a newly independent held-out evaluation. No v2 checkpoints are loaded or altered.

**IMPORTANT Kaggle note:** Sessions can end and lose interactive output. Back up v3 checkpoints to durable storage at sensible intervals. Kaggle Notebook versions and output datasets are helpful, but merely saving notebook source does not guarantee interrupted checkpoints survive.

## 1 — GPU checks and settings
In Notebook settings select **GPU T4 x2** if available (this script uses CUDA device 0 only; two 16-GB cards do not form one 32-GB memory pool). Enable **Internet** for GitHub/pip/ImageNet weights, where permitted. Add a private Kaggle Dataset containing your unchanged `data.zip`.

In [ ]:
import os, sys, subprocess, torch
from pathlib import Path
assert torch.cuda.is_available(), 'Kaggle Settings → Accelerator → GPU needed'
print('Torch',torch.__version__,'GPU count',torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props=torch.cuda.get_device_properties(i)
    print('GPU',i,props.name,round(props.total_memory / 1024**3,2),'GiB')
subprocess.run(['nvidia-smi'],check=True)

## 2 — Dependencies
`torch` and `torchvision` come from the Kaggle image; install `timm` and required helpers. Record exact versions in run metadata. If model weights cannot download, STOP and attach the exact pretrained weights—do not fall back to random initialization.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','timm','pyyaml','scikit-learn','pandas','matplotlib'],check=True)
import timm, yaml, torchvision, sklearn, pandas as pd, matplotlib.pyplot as plt
print('timm',timm.__version__,'torchvision',torchvision.__version__,'sklearn',sklearn.__version__)

## 3 — Clone and pin the v3 GitHub commit
From VS Code, commit and push exactly the three new v3 files first. Replace the placeholder below with the resulting `git rev-parse HEAD` output. This v3 commit **must differ** from frozen Notebook 11 commit `91e320a506749d01d141ff2115ecb585e03bce31`.

In [ ]:
REPO_URL='https://github.com/Batsy1857/vehicle-make-model-recognition.git'
V3_BRANCH='experiment/full-finetuning-v3'  # pushed branch containing new files
FROZEN_V3_COMMIT='REPLACE_WITH_V3_COMMIT_SHA'  # mandatory
assert len(FROZEN_V3_COMMIT)==40 and all(c in '0123456789abcdef' for c in FROZEN_V3_COMMIT), 'Paste actual 40-digit git SHA first'
assert FROZEN_V3_COMMIT != '91e320a506749d01d141ff2115ecb585e03bce31'
REPO=Path('/kaggle/working/vehicle-make-model-recognition')
if not REPO.exists():
    subprocess.run(['git','clone',REPO_URL,str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin',V3_BRANCH],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',FROZEN_V3_COMMIT],check=True)
sha=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert sha==FROZEN_V3_COMMIT
for rel in ['src/train_vehicle_fullft.py','configs/full_finetuning.yaml','notebooks/12_full_finetuning_kaggle.ipynb']:
    assert (REPO/rel).exists(),f'Missing new v3 file: {rel}'
print('PINNED NEW V3 CODE:',sha)

## 4 — Restore EXACT prepared dataset
Use Kaggle **Add Input** to attach your own `data.zip` as a private dataset. The zip must contain paths such as `data/pakistan/` and `data/stanford_cars/`. Do not re-download or split images.

In [ ]:
import zipfile
archives=list(Path('/kaggle/input').rglob('data.zip'))
assert len(archives)==1, f'Expected exactly one uploaded data.zip; found: {archives}'
DATA_ZIP=archives[0]
print('Using',DATA_ZIP,'size GiB',round(DATA_ZIP.stat().st_size/1024**3,2))
if not (REPO/'data/pakistan').is_dir() or not (REPO/'data/stanford_cars').is_dir():
    with zipfile.ZipFile(DATA_ZIP) as archive:
        names=archive.namelist()
        assert any(n.startswith('data/pakistan/') for n in names)
        assert any(n.startswith('data/stanford_cars/') for n in names)
        archive.extractall(REPO)
print('Data restored under',REPO/'data')

## 5 — Frozen manifest audit and recipe validation (no test evaluation)
Dataset files are checked for existence, but the test set is not loaded by the training code.

In [ ]:
sys.path.insert(0,str(REPO/'src'))
from vehicle_data import inspect_manifest
from train_vehicle_fullft import configure_v3_stage, epoch_lr
cfg=yaml.safe_load((REPO/'configs/experiments.yaml').read_text())
recipe=yaml.safe_load((REPO/'configs/full_finetuning.yaml').read_text())
for name in ('pakistan','stanford_cars'):
    audit=inspect_manifest(REPO,name,check_files=True)
    assert audit['manifest_sha256']==cfg['datasets'][name]['manifest_sha256']
    for split,expected in cfg['datasets'][name]['verified_split_counts'].items():
        observed='excluded_test_overlap' if name=='pakistan' and split=='excluded' else split
        assert audit['counts'].get(observed,0)==expected
    print(name,audit['counts'],'SHA PASS')
assert recipe['seed']==42
assert recipe['stages']['head_only']['epochs']==5
assert recipe['stages']['full_fine_tune']['epochs']['stanford_cars']==40
print('Frozen split + v3 recipe audit passed')

## 6 — Inspect planned learning-rate schedule

In [ ]:
ft=recipe['stages']['full_fine_tune']
rate=[epoch_lr(i,40,ft['peak_learning_rate'],ft['minimum_learning_rate'],
               ft['warmup_epochs'],ft['warmup_start_fraction']) for i in range(40)]
plt.figure(figsize=(8,3));plt.plot(range(1,41),rate,marker='.',ms=3)
plt.xlabel('Fine-tune epoch');plt.ylabel('Learning rate');plt.title('Warm-up + cosine decay (planned)');plt.grid(alpha=.2);plt.show()
print('First / peak / last LR:',rate[0],max(rate),rate[-1])

## 7 — Mandatory GPU-memory preflight: E3, then E4
This uses **synthetic** 16-image batches and two full-backbone AdamW update steps, so the memory count includes optimizer states. It does **not** load pretrained weights or evaluate data. If it fails or reserves at least 90% of GPU RAM, do not start batch-16 v3 training. Stop and document feasibility rather than secretly changing the protocol.

In [ ]:
OUTPUT=Path('/kaggle/working/v3_runs')
OUTPUT.mkdir(parents=True,exist_ok=True)
for exp in ('E3','E4'):
    args=[sys.executable,str(REPO/'src/train_vehicle_fullft.py'),'--experiment',exp,
          '--run-id',f'{exp}-v3-memory','--output-dir',str(OUTPUT),'--mode','memory']
    print('MEMORY PREFLIGHT',exp,flush=True)
    subprocess.run(args,cwd=REPO,check=True)
print('Memory checks passed for E3 and E4')

## 8 — Mandatory smoke tests: E3, then E4
Each smoke job does 1 mini head-only epoch and 1 mini full-backbone epoch (2 train + 2 validation batches each), writing to `E3-v3-smoke01` and `E4-v3-smoke01`. These are engineering checks, **not** reportable performance measurements. New ImageNet pretrained initialization is required here.

In [ ]:
for exp in ('E3','E4'):
    rid=f'{exp}-v3-smoke01'
    args=[sys.executable,str(REPO/'src/train_vehicle_fullft.py'),'--experiment',exp,
          '--run-id',rid,'--output-dir',str(OUTPUT),'--smoke']
    print('SMOKE',exp,flush=True)
    subprocess.run(args,cwd=REPO,check=True)
    m=__import__('json').loads((OUTPUT/rid/'metadata.json').read_text())
    assert m['status']=='smoke_completed'
    assert m['parameters_full_fine_tune']['frozen']==0
    print('SMOKE OK:',rid,'trainable',m['parameters_full_fine_tune']['trainable'])

## 9 — Full experiments (manual, one at a time)
**Default: disabled.** Only enable after BOTH memory checks and smoke tests pass. Run E3 first, then E4. Use unique run IDs. Use `RESUME=True` only when both v3 metadata and last.pt survived and the original pinned SHA is unchanged.

⚠️ Kaggle sessions are finite. Save durable copies of checkpoints between sessions; unsaved files in `/kaggle/working` may disappear. Do not try to claim a run resumed if its optimizer/RNG state is missing.

In [ ]:
RUN_TRAINING=False
EXPERIMENT='E3'       # E3 first, E4 second
RUN_ID=f'{EXPERIMENT}-v3-run01'
RESUME=False
if RUN_TRAINING:
    cmd=[sys.executable,str(REPO/'src/train_vehicle_fullft.py'),
         '--experiment',EXPERIMENT,'--run-id',RUN_ID,'--output-dir',str(OUTPUT)]
    if RESUME:cmd.append('--resume')
    subprocess.run(cmd,cwd=REPO,check=True)
    print('FINISHED',OUTPUT/RUN_ID)
else:
    print('Full training intentionally disabled; set RUN_TRAINING=True after preflight.')

## 10 — Validation-only convergence and comparison
No test images are touched. Compare both model/strategy results only after their checkpoint selections are frozen.

In [ ]:
import json
rows=[]
for exp in ('E3','E4','E1','E2'):
    d=OUTPUT/f'{exp}-v3-run01'
    if not (d/'metadata.json').is_file():continue
    meta=json.loads((d/'metadata.json').read_text())
    if meta['status']!='completed':continue
    rows.append({'experiment':exp,'kind':meta['kind'],
      'best_val_macro_f1':meta['best_validation_macro_f1'],
      'selected_stage':meta['selected_stage'], 'selected_epoch':meta['selected_epoch'],
      'head_epochs':meta['epochs_completed_head_only'],
      'full_epochs':meta['epochs_completed_finetune'],
      'full_trainable_params':meta['parameters_full_fine_tune']['trainable'],
      'train_seconds_excl_validation':meta['total_train_seconds_excluding_validation'],
      'peak_gpu_memory_mb':meta['peak_gpu_memory_mb'],
      'git_sha':meta['git_commit']})
display(pd.DataFrame(rows))

## 11 — Optional single post-hoc test evaluation (disabled)
Only perform **once**, after validation-based selection is final and the exploratory nature is documented. Do not use v3 test scores to revise settings and rerun. The code refuses an existing `test_metrics.json`.

In [ ]:
RUN_POSTHOC_TEST=False
if RUN_POSTHOC_TEST:
    for exp in ('E3','E4'):
        rid=f'{exp}-v3-run01'
        assert json.loads((OUTPUT/rid/'metadata.json').read_text())['status']=='completed'
    for exp in ('E3','E4'):
        cmd=[sys.executable,str(REPO/'src/train_vehicle_fullft.py'),
             '--experiment',exp,'--run-id',f'{exp}-v3-run01',
             '--output-dir',str(OUTPUT),'--mode','test','--acknowledge-posthoc-test']
        subprocess.run(cmd,cwd=REPO,check=True)
else:
    print('Post-hoc test disabled (recommended while exploring validation curves).')

## 12 — Inspect output and back up files
Each completed v3 run has `metadata.json`, `history.csv`, `best.pt`, `best_head.pt`, `best_finetune.pt`, and `last.pt`. **Back up checkpoints to persistent Kaggle outputs / an output Dataset or download them before the session expires.** Do not upload `.pt` files to GitHub. Export small CSV/JSON for the report.

For automated Kaggle `Save & Run All`, configure a single experimental run that can finish within Kaggle limits; the save executes in a new clean session, not the current interactive session.

In [ ]:
for d in sorted(OUTPUT.glob('E*-v3-*')):
    if d.is_dir():
        print(d.name,'files:',[(f.name,round(f.stat().st_size/1024**2,1)) for f in d.iterdir() if f.is_file()])
# Optional: generate a SMALL summary archive (excluding huge checkpoints)
import zipfile
archive_path=Path('/kaggle/working/v3_logs_only.zip')
with zipfile.ZipFile(archive_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in OUTPUT.rglob('*'):
        if f.is_file() and f.suffix.lower() in ('.csv','.json'):
            z.write(f,arcname=f.relative_to(OUTPUT))
print('Small metrics archive:',archive_path)

## Reporting limitations

- The **v2** test values are previously inspected: E3 53.04% / macro-F1 0.5305, E4 66.65% / macro-F1 0.6625 (also E1 80.83% / 0.7472; E2 87.88% / 0.8211).
- The **v3** results are not available until training and evaluation finish.
- Higher v3 scores, if observed, are exploratory/post-hoc comparisons and do not establish a statistically significant benefit from one seed.
- If batch 16 does not fit T4 GPU, preserve fairness by reporting the memory limitation instead of quietly changing it.